# import liberaries

In [ ]:
%%capture
!pip install torch accelerate transformers diffusers safetensors bitsandbytes
!pip install diffusers transformers accelerate safetensors

In [ ]:
import torch
from diffusers import BitsAndBytesConfig, SD3Transformer2DModel, StableDiffusion3Pipeline
from diffusers import StableDiffusion3Pipeline
from diffusers import DiffusionPipeline
import torch
import gc

In [ ]:
torch.cuda.empty_cache()
gc.collect()

60

# Connect to huggingface

In [ ]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) y
Token is valid (permission: write

# Load and Quantize the Model

 ### Configure the 4-bit Quantization

In [ ]:
nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

### Load the Model in 4-bit

In [ ]:
model_id = "stabilityai/stable-diffusion-3.5-medium"

# Load the 2D Transformer part with 4-bit quantization
model_nf4 = SD3Transformer2DModel.from_pretrained(
    model_id,
    subfolder="transformer",
    quantization_config=nf4_config,
    torch_dtype=torch.float16
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Build the Pipeline

In [ ]:
pipeline = StableDiffusion3Pipeline.from_pretrained(
    model_id,
    transformer=model_nf4,
    torch_dtype=torch.float16
)

# Offload model layers not in use to the CPU
pipeline.enable_model_cpu_offload()

# Sliced attention: helps reduce VRAM usage on moderate GPUs
pipeline.enable_attention_slicing()

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running them will fail. Please make sure to use an accelerator to run the pipeline in inference, due to the lack of support for`float16` operations on this device in PyTorch. Please, remove the `torch_dtype=torch.float16` argument, or use another device for inference.
Pipelines loaded with `dtype=torch.float16` cannot run with `cpu` device. It is not recommended to move them to `cpu` as running th

StableDiffusion3Pipeline {
  "_class_name": "StableDiffusion3Pipeline",
  "_diffusers_version": "0.32.2",
  "_name_or_path": "stabilityai/stable-diffusion-3.5-medium",
  "feature_extractor": [
    null,
    null
  ],
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "FlowMatchEulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModelWithProjection"
  ],
  "text_encoder_2": [
    "transformers",
    "CLIPTextModelWithProjection"
  ],
  "text_encoder_3": [
    "transformers",
    "T5EncoderModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_3": [
    "transformers",
    "T5TokenizerFast"
  ],
  "transformer": [
    "diffusers",
    "SD3Transformer2DModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

### Run the Pipeline with Custom Prompts

In [ ]:
prompt = "A neon-lit futuristic cityscape at sunset, ultra-detailed, cinematic lighting"

# image = pipeline(
#     prompt=prompt,
#     num_inference_steps=30,
#     guidance_scale=4.5,
#     max_sequence_length=256
# ).images[0]
image = pipeline(
    prompt=prompt,
    num_inference_steps=30,
    guidance_scale=4.5,
    max_sequence_length=256,
    height=256,
    width=256
).images[0]


image.save("generated_image_1.png")


  0%|          | 0/30 [00:00<?, ?it/s]

RuntimeError: All input tensors need to be on the same GPU, but found some tensors to not be on a GPU:
 [(torch.Size([1, 196608]), device(type='cpu')), (torch.Size([6144]), device(type='cpu')), (torch.Size([1536, 256]), device(type='cpu'))]

# another version

very light for gpu

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe_small = DiffusionPipeline.from_pretrained("OFA-Sys/small-stable-diffusion-v0")
pipe_small.to(device)

prompt_small = "A lovely painting of a small mountain cabin by the lake, 4k quality"
image_small = pipe_small(prompt_small).images[0]
image_small.save("not_as_good_generated_image.png")

print("Done generating images.")